In [2]:
# Run only if these packages are unavailable.
! pip install -q "zarr<3" numcodecs rasterio xarray tqdm

In [3]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import shutil
import time

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import xarray as xr
import zarr
from numcodecs import Blosc
from tqdm.auto import tqdm

/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_DIR = Path("../datasets/VNP46")
OUTPUT_DIR = DATA_DIR / "processed"

A1_STORE = OUTPUT_DIR / "Haiyan_VNP46A1.zarr"
A2_STORE = OUTPUT_DIR / "Haiyan_VNP46A2.zarr"

MANIFEST_FILE = OUTPUT_DIR / "source_manifest.csv"
PILOT_QC_FILE = OUTPUT_DIR / "pilot_qc.csv"
PROCESSING_LOG_FILE = OUTPUT_DIR / "processing_log.csv"
PILOT_PASS_FILE = OUTPUT_DIR / "pilot_qc_passed.json"


# ------------------------------------------------------------
# Processing settings
# ------------------------------------------------------------

EVENT_DATE = pd.Timestamp("2013-11-08")

PILOT_DAYS = 5

CHUNK_Y = 512
CHUNK_X = 512

QC_WINDOW_SIZE = 128
QC_SAMPLE_STEP = 8

EXTRA_FILL_VALUES = (-9999.0, -999.0)

EXPECTED_PRODUCTS = ("VNP46A1", "VNP46A2")
EXPECTED_FILES_PER_PRODUCT = 3

STRICT_DAILY_COVERAGE = True
REQUIRE_MATCHING_PRODUCT_DATES = True

# Keep False unless intentionally rebuilding both stores.
RESET_STORES = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if int(zarr.__version__.split(".")[0]) != 2:
    raise RuntimeError(
        f"Detected zarr {zarr.__version__}. "
        'Install the stable v2 API with: %pip install "zarr<3"'
    )

In [7]:
YMD_PATTERN = re.compile(
    r"(?<!\d)(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d)(?!\d)"
)

DOY_PATTERN = re.compile(
    r"(?<!\d)A?(20\d{2})[-_]?([0-3]\d{2})(?!\d)"
)


def identify_product(path):
    match = re.search(r"VNP46A[12]", path.name.upper())

    if match is None:
        raise ValueError(f"Cannot identify product from filename: {path.name}")

    return match.group(0)


def parse_date_and_variable(label, band_index):
    label = str(label).strip()

    ymd_match = YMD_PATTERN.search(label)

    if ymd_match is not None:
        year, month, day = ymd_match.groups()
        date = pd.to_datetime(
            f"{year}-{month}-{day}",
            errors="coerce",
        )
        date_span = ymd_match.span()

    else:
        doy_match = DOY_PATTERN.search(label)

        if doy_match is None:
            return pd.NaT, None

        year, day_of_year = doy_match.groups()
        date = pd.to_datetime(
            f"{year}{day_of_year}",
            format="%Y%j",
            errors="coerce",
        )
        date_span = doy_match.span()

    if pd.isna(date):
        return pd.NaT, None

    variable = (
        label[:date_span[0]]
        + "_"
        + label[date_span[1]:]
    )

    variable = re.sub(
        r"(?<![A-Za-z0-9])VNP46A[12](?![A-Za-z0-9])",
        "_",
        variable,
        flags=re.IGNORECASE,
    )

    variable = YMD_PATTERN.sub("_", variable)
    variable = DOY_PATTERN.sub("_", variable)

    variable = re.sub(r"[^A-Za-z0-9_]+", "_", variable)
    variable = re.sub(r"_+", "_", variable).strip("_")

    if not variable:
        variable = f"band_{band_index:04d}"

    if variable[0].isdigit():
        variable = f"band_{variable}"

    if variable in {"date", "x", "y", "processed", "spatial_ref"}:
        variable = f"source_{variable}"

    return pd.Timestamp(date).normalize(), variable


source_files = sorted(
    set(DATA_DIR.glob("*.tif"))
    | set(DATA_DIR.glob("*.tiff"))
)

if not source_files:
    raise FileNotFoundError(
        f"No GeoTIFF files found in {DATA_DIR.resolve()}"
    )

print(f"Source directory: {DATA_DIR.resolve()}")
print(f"GeoTIFF files found: {len(source_files)}")

for path in source_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**2:,.1f} MB")

Source directory: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46
GeoTIFF files found: 6
  Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif: 407.9 MB
  Haiyan_VNP46A1_KeyBands_PostEvent_D181_D365.tif: 410.3 MB
  Haiyan_VNP46A1_KeyBands_PreEvent_D180_D1.tif: 397.2 MB
  Haiyan_VNP46A2_AllBands_PostEvent_D0_D180.tif: 228.8 MB
  Haiyan_VNP46A2_AllBands_PostEvent_D181_D365.tif: 208.2 MB
  Haiyan_VNP46A2_AllBands_PreEvent_D180_D1.tif: 196.1 MB


In [8]:
manifest_records = []
file_records = []

for path in source_files:
    product = identify_product(path)

    with rasterio.open(path) as src:
        transform_values = tuple(src.transform)[:6]

        file_records.append(
            {
                "product": product,
                "source_file": str(path),
                "source_name": path.name,
                "size_mb": path.stat().st_size / 1024**2,
                "band_count": src.count,
                "height": src.height,
                "width": src.width,
                "crs": src.crs.to_wkt() if src.crs else None,
                "transform": transform_values,
            }
        )

        for band_index in src.indexes:
            description = src.descriptions[band_index - 1]
            band_tags = src.tags(band_index)

            label = (
                description
                or band_tags.get("long_name")
                or band_tags.get("DESCRIPTION")
                or band_tags.get("description")
                or f"band_{band_index:04d}"
            )

            date, variable = parse_date_and_variable(
                label,
                band_index,
            )

            manifest_records.append(
                {
                    "product": product,
                    "source_file": str(path),
                    "source_name": path.name,
                    "band_index": band_index,
                    "description": label,
                    "date": date,
                    "variable": variable,
                    "dtype": src.dtypes[band_index - 1],
                    "nodata": src.nodatavals[band_index - 1],
                    "tags_json": json.dumps(
                        band_tags,
                        sort_keys=True,
                        default=str,
                    ),
                }
            )

manifest = pd.DataFrame(manifest_records)
file_inventory = pd.DataFrame(file_records)

manifest["date"] = pd.to_datetime(manifest["date"])

display(
    file_inventory[
        [
            "product",
            "source_name",
            "size_mb",
            "band_count",
            "height",
            "width",
        ]
    ]
)

display(manifest.head(20))

,product,source_name,size_mb,band_count,height,width
0,VNP46A1,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,407.902714,1448,674,473
1,VNP46A1,Haiyan_VNP46A1_KeyBands_PostEvent_D181_D365.tif,410.272547,1480,674,473
2,VNP46A1,Haiyan_VNP46A1_KeyBands_PreEvent_D180_D1.tif,397.180687,1440,674,473
3,VNP46A2,Haiyan_VNP46A2_AllBands_PostEvent_D0_D180.tif,228.797158,1267,674,473
4,VNP46A2,Haiyan_VNP46A2_AllBands_PostEvent_D181_D365.tif,208.152193,1295,674,473
5,VNP46A2,Haiyan_VNP46A2_AllBands_PreEvent_D180_D1.tif,196.066198,1260,674,473


,product,source_file,source_name,band_index,description,date,variable,dtype,nodata,tags_json
0,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,1,2013_11_08_DNB_At_Sensor_Radiance_500m,2013-11-08,DNB_At_Sensor_Radiance_500m,float32,None,{}
1,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,2,2013_11_08_Sensor_Zenith,2013-11-08,Sensor_Zenith,float32,None,{}
2,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,3,2013_11_08_Sensor_Azimuth,2013-11-08,Sensor_Azimuth,float32,None,{}
3,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,4,2013_11_08_UTC_Time,2013-11-08,UTC_Time,float32,None,{}
4,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,5,2013_11_08_Granule,2013-11-08,Granule,float32,None,{}
5,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,6,2013_11_08_QF_DNB,2013-11-08,QF_DNB,float32,None,{}
6,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,7,2013_11_08_QF_Cloud_Mask,2013-11-08,QF_Cloud_Mask,float32,None,{}
7,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,8,2013_11_08_Moon_Illumination_Fraction,2013-11-08,Moon_Illumination_Fraction,float32,None,{}
8,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,9,2013_11_09_DNB_At_Sensor_Radiance_500m,2013-11-09,DNB_At_Sensor_Radiance_500m,float32,None,{}
9,VNP46A1,../datasets/VNP46/Haiyan_VNP46A1_KeyBands_Post...,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,10,2013_11_09_Sensor_Zenith,2013-11-09,Sensor_Zenith,float32,None,{}


In [9]:
# ------------------------------------------------------------
# 1. Expected products and file counts
# ------------------------------------------------------------

products_found = tuple(sorted(manifest["product"].unique()))

missing_products = sorted(
    set(EXPECTED_PRODUCTS) - set(products_found)
)

if missing_products:
    raise ValueError(
        f"Missing expected products: {missing_products}"
    )

file_counts = (
    file_inventory.groupby("product")["source_name"]
    .nunique()
    .sort_index()
)

print("Source files per product:")
display(file_counts.rename("files").to_frame())

unexpected_file_counts = file_counts[
    file_counts != EXPECTED_FILES_PER_PRODUCT
]

if not unexpected_file_counts.empty:
    raise ValueError(
        "Expected three period files per product, but found:\n"
        f"{unexpected_file_counts}"
    )


# ------------------------------------------------------------
# 2. Date parsing
# ------------------------------------------------------------

unparsed = manifest[manifest["date"].isna()].copy()

if not unparsed.empty:
    display(
        unparsed[
            [
                "product",
                "source_name",
                "band_index",
                "description",
            ]
        ].head(30)
    )

    raise ValueError(
        f"Could not parse dates for {len(unparsed):,} bands. "
        "Inspect the displayed raw band descriptions before processing."
    )


# ------------------------------------------------------------
# 3. Duplicate product-date-variable records
# ------------------------------------------------------------

duplicate_mask = manifest.duplicated(
    subset=["product", "date", "variable"],
    keep=False,
)

duplicates = manifest.loc[
    duplicate_mask,
    [
        "product",
        "date",
        "variable",
        "source_name",
        "band_index",
    ],
].sort_values(["product", "date", "variable"])

if not duplicates.empty:
    display(duplicates.head(50))

    raise ValueError(
        "Duplicate product-date-variable records detected. "
        "This may indicate overlapping export periods."
    )


# ------------------------------------------------------------
# 4. Consistent grid within each product
# ------------------------------------------------------------

product_grids = {}

for product, product_files in file_inventory.groupby("product"):
    reference = product_files.iloc[0]

    for _, row in product_files.iloc[1:].iterrows():
        same_shape = (
            row["height"] == reference["height"]
            and row["width"] == reference["width"]
        )

        same_crs = row["crs"] == reference["crs"]

        same_transform = np.allclose(
            np.asarray(row["transform"], dtype=float),
            np.asarray(reference["transform"], dtype=float),
            rtol=0,
            atol=1e-10,
        )

        if not (same_shape and same_crs and same_transform):
            raise ValueError(
                f"{product} grid mismatch:\n"
                f"Reference: {reference['source_name']}\n"
                f"Mismatch:  {row['source_name']}"
            )

    product_grids[product] = {
        "height": int(reference["height"]),
        "width": int(reference["width"]),
        "crs_wkt": reference["crs"],
        "transform": tuple(reference["transform"]),
    }


# ------------------------------------------------------------
# 5. Consistent variables across dates
# ------------------------------------------------------------

for product, product_manifest in manifest.groupby("product"):
    variable_sets = (
        product_manifest.groupby("date")["variable"]
        .apply(lambda values: tuple(sorted(values)))
    )

    reference_variables = variable_sets.iloc[0]

    inconsistent_dates = variable_sets[
        variable_sets != reference_variables
    ]

    if not inconsistent_dates.empty:
        display(inconsistent_dates.head(20))

        raise ValueError(
            f"{product} does not contain the same bands on every date."
        )


# ------------------------------------------------------------
# 6. Daily continuity
# ------------------------------------------------------------

for product, product_manifest in manifest.groupby("product"):
    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    expected_dates = pd.date_range(
        dates.min(),
        dates.max(),
        freq="D",
    )

    missing_dates = expected_dates.difference(dates)

    print(
        f"{product}: {len(dates):,} dates | "
        f"{dates.min().date()} to {dates.max().date()} | "
        f"{len(missing_dates):,} missing calendar dates"
    )

    if len(missing_dates):
        print(
            "First missing dates:",
            [date.strftime("%Y-%m-%d") for date in missing_dates[:20]],
        )

        if STRICT_DAILY_COVERAGE:
            raise ValueError(
                f"{product} has {len(missing_dates):,} missing dates."
            )


# ------------------------------------------------------------
# 7. Matching A1 and A2 date coverage
# ------------------------------------------------------------

date_sets = {
    product: set(
        pd.DatetimeIndex(
            product_manifest["date"].unique()
        )
    )
    for product, product_manifest
    in manifest.groupby("product")
}

if REQUIRE_MATCHING_PRODUCT_DATES:
    a1_only = sorted(
        date_sets["VNP46A1"] - date_sets["VNP46A2"]
    )

    a2_only = sorted(
        date_sets["VNP46A2"] - date_sets["VNP46A1"]
    )

    if a1_only or a2_only:
        print("Dates found only in VNP46A1:", a1_only[:20])
        print("Dates found only in VNP46A2:", a2_only[:20])

        raise ValueError(
            "VNP46A1 and VNP46A2 do not cover identical dates."
        )


# ------------------------------------------------------------
# 8. Haiyan date availability
# ------------------------------------------------------------

for product, dates in date_sets.items():
    if EVENT_DATE not in dates:
        raise ValueError(
            f"Haiyan date {EVENT_DATE.date()} is absent from {product}."
        )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

source_summary = (
    manifest.groupby("product")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        dates=("date", "nunique"),
        variables=("variable", "nunique"),
        source_bands=("band_index", "size"),
    )
)

display(source_summary)

for product, product_manifest in manifest.groupby("product"):
    variables = sorted(product_manifest["variable"].unique())

    print(f"\n{product} variables:")
    for variable in variables:
        print(f"  {variable}")

manifest.to_csv(MANIFEST_FILE, index=False)

print("\nSOURCE SANITY CHECKS: PASSED")
print(f"Manifest saved: {MANIFEST_FILE}")

Source files per product:


,files
product,
VNP46A1,3
VNP46A2,3


VNP46A1: 546 dates | 2013-05-12 to 2014-11-08 | 0 missing calendar dates
VNP46A2: 546 dates | 2013-05-12 to 2014-11-08 | 0 missing calendar dates


,start_date,end_date,dates,variables,source_bands
product,,,,,
VNP46A1,2013-05-12,2014-11-08,546,8,4368
VNP46A2,2013-05-12,2014-11-08,546,7,3822



VNP46A1 variables:
  DNB_At_Sensor_Radiance_500m
  Granule
  Moon_Illumination_Fraction
  QF_Cloud_Mask
  QF_DNB
  Sensor_Azimuth
  Sensor_Zenith
  UTC_Time

VNP46A2 variables:
  DNB_BRDF_Corrected_NTL
  DNB_Lunar_Irradiance
  Gap_Filled_DNB_BRDF_Corrected_NTL
  Latest_High_Quality_Retrieval
  Mandatory_Quality_Flag
  QF_Cloud_Mask
  Snow_Flag

SOURCE SANITY CHECKS: PASSED
Manifest saved: ../datasets/VNP46/processed/source_manifest.csv


In [10]:
COMPRESSOR = Blosc(
    cname="zstd",
    clevel=5,
    shuffle=Blosc.BITSHUFFLE,
)


def manifest_signature(product, product_manifest, grid):
    signature_table = product_manifest[
        [
            "source_name",
            "band_index",
            "date",
            "variable",
        ]
    ].copy()

    signature_table["date"] = (
        signature_table["date"].dt.strftime("%Y-%m-%d")
    )

    payload = {
        "product": product,
        "grid": {
            "height": grid["height"],
            "width": grid["width"],
            "crs_wkt": grid["crs_wkt"],
            "transform": list(grid["transform"]),
        },
        "manifest": signature_table.to_dict("records"),
    }

    encoded = json.dumps(
        payload,
        sort_keys=True,
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def initialise_store(
    product,
    product_manifest,
    grid,
    store_path,
):
    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    variables = sorted(
        product_manifest["variable"].unique()
    )

    height = grid["height"]
    width = grid["width"]

    transform = grid["transform"]
    a, b, c, d, e, f = transform

    if not np.isclose(b, 0) or not np.isclose(d, 0):
        raise ValueError(
            f"{product} uses a rotated grid. "
            "One-dimensional x/y coordinates are not valid."
        )

    x = c + a * (np.arange(width) + 0.5)
    y = f + e * (np.arange(height) + 0.5)

    signature = manifest_signature(
        product,
        product_manifest,
        grid,
    )

    if RESET_STORES and store_path.exists():
        shutil.rmtree(store_path)

    if store_path.exists():
        root = zarr.open_group(
            str(store_path),
            mode="a",
        )

        existing_signature = root.attrs.get(
            "manifest_signature"
        )

        if existing_signature != signature:
            raise ValueError(
                f"{store_path.name} exists but does not match "
                "the current source manifest. Set RESET_STORES = True "
                "only if the existing processed store can be replaced."
            )

        print(f"Using existing store: {store_path}")
        return root

    root = zarr.open_group(
        str(store_path),
        mode="w",
    )

    root.attrs.update(
        {
            "product": product,
            "title": f"Haiyan {product} daily raster cube",
            "created_utc": datetime.utcnow().isoformat(),
            "manifest_signature": signature,
            "crs_wkt": grid["crs_wkt"],
            "transform": list(transform),
            "height": height,
            "width": width,
            "date_start": dates.min().strftime("%Y-%m-%d"),
            "date_end": dates.max().strftime("%Y-%m-%d"),
            "date_count": len(dates),
            "variables": variables,
            "chunk_shape": [1, CHUNK_Y, CHUNK_X],
            "missing_value": "NaN",
            "scaling_applied": False,
        }
    )

    date_array = root.create_dataset(
        "date",
        data=dates.values.astype("datetime64[ns]"),
        chunks=(min(len(dates), 1024),),
        compressor=None,
    )
    date_array.attrs["_ARRAY_DIMENSIONS"] = ["date"]

    x_array = root.create_dataset(
        "x",
        data=x.astype(np.float64),
        chunks=(min(width, 4096),),
        compressor=None,
    )
    x_array.attrs["_ARRAY_DIMENSIONS"] = ["x"]

    y_array = root.create_dataset(
        "y",
        data=y.astype(np.float64),
        chunks=(min(height, 4096),),
        compressor=None,
    )
    y_array.attrs["_ARRAY_DIMENSIONS"] = ["y"]

    processed_array = root.create_dataset(
        "processed",
        data=np.zeros(len(dates), dtype=np.uint8),
        chunks=(min(len(dates), 1024),),
        dtype="u1",
        fill_value=255,
        compressor=None,
    )
    processed_array.attrs.update(
        {
            "_ARRAY_DIMENSIONS": ["date"],
            "flag_values": [0, 1],
            "flag_meanings": "not_processed processed",
        }
    )

    spatial_ref = root.create_dataset(
        "spatial_ref",
        shape=(),
        dtype="i4",
        fill_value=0,
        compressor=None,
    )
    spatial_ref[()] = 0
    spatial_ref.attrs.update(
        {
            "_ARRAY_DIMENSIONS": [],
            "spatial_ref": grid["crs_wkt"],
            "crs_wkt": grid["crs_wkt"],
            "GeoTransform": f"{c} {a} {b} {f} {d} {e}",
        }
    )

    for variable in variables:
        variable_rows = product_manifest[
            product_manifest["variable"].eq(variable)
        ]

        source_descriptions = sorted(
            variable_rows["description"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        source_tags = (
            variable_rows["tags_json"]
            .dropna()
            .astype(str)
            .iloc[0]
        )

        array = root.create_dataset(
            variable,
            shape=(len(dates), height, width),
            chunks=(
                1,
                min(CHUNK_Y, height),
                min(CHUNK_X, width),
            ),
            dtype="f4",
            fill_value=np.nan,
            compressor=COMPRESSOR,
            dimension_separator="/",
        )

        array.attrs.update(
            {
                "_ARRAY_DIMENSIONS": ["date", "y", "x"],
                "grid_mapping": "spatial_ref",
                "source_descriptions": source_descriptions,
                "source_tags_example": source_tags,
                "scaling_applied": False,
            }
        )

    print(f"Created store: {store_path}")
    print(
        f"  Shape per variable: "
        f"({len(dates):,}, {height:,}, {width:,})"
    )
    print(f"  Variables: {len(variables)}")

    return root


STORE_PATHS = {
    "VNP46A1": A1_STORE,
    "VNP46A2": A2_STORE,
}

stores = {}

for product in EXPECTED_PRODUCTS:
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    stores[product] = initialise_store(
        product=product,
        product_manifest=product_manifest,
        grid=product_grids[product],
        store_path=STORE_PATHS[product],
    )

Using existing store: ../datasets/VNP46/processed/Haiyan_VNP46A1.zarr
Using existing store: ../datasets/VNP46/processed/Haiyan_VNP46A2.zarr


In [11]:
def iter_windows(height, width, chunk_y, chunk_x):
    for row_off in range(0, height, chunk_y):
        window_height = min(
            chunk_y,
            height - row_off,
        )

        for col_off in range(0, width, chunk_x):
            window_width = min(
                chunk_x,
                width - col_off,
            )

            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=window_width,
                height=window_height,
            )


def clean_source_block(block):
    data = np.asarray(
        block.filled(np.nan),
        dtype=np.float32,
    )

    for fill_value in EXTRA_FILL_VALUES:
        data[data == fill_value] = np.nan

    data[~np.isfinite(data)] = np.nan

    return data


def process_dates(
    product,
    dates_to_process,
    skip_processed=True,
):
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    date_lookup = {
        pd.Timestamp(date): index
        for index, date in enumerate(dates)
    }

    root = zarr.open_group(
        str(STORE_PATHS[product]),
        mode="a",
    )

    height = int(root.attrs["height"])
    width = int(root.attrs["width"])

    records = []

    for date in tqdm(
        pd.DatetimeIndex(dates_to_process),
        desc=product,
    ):
        date = pd.Timestamp(date).normalize()
        date_index = date_lookup[date]

        if (
            skip_processed
            and root["processed"][date_index] == 1
        ):
            continue

        date_rows = product_manifest[
            product_manifest["date"].eq(date)
        ].sort_values("variable")

        if date_rows.empty:
            raise KeyError(
                f"No {product} source bands found for {date.date()}."
            )

        start_time = time.perf_counter()

        try:
            for source_file, file_rows in date_rows.groupby(
                "source_file",
                sort=False,
            ):
                file_rows = file_rows.sort_values("variable")

                band_indexes = (
                    file_rows["band_index"]
                    .astype(int)
                    .tolist()
                )

                variables = file_rows["variable"].tolist()

                with rasterio.open(source_file) as src:
                    for window in iter_windows(
                        height,
                        width,
                        CHUNK_Y,
                        CHUNK_X,
                    ):
                        block = src.read(
                            band_indexes,
                            window=window,
                            masked=True,
                            out_dtype="float32",
                        )

                        block = clean_source_block(block)

                        row_start = int(window.row_off)
                        row_stop = row_start + int(window.height)

                        col_start = int(window.col_off)
                        col_stop = col_start + int(window.width)

                        for band_position, variable in enumerate(
                            variables
                        ):
                            root[variable][
                                date_index,
                                row_start:row_stop,
                                col_start:col_stop,
                            ] = block[band_position]

            # Mark complete only after every band and window succeeds.
            root["processed"][date_index] = np.uint8(1)

        except Exception:
            root["processed"][date_index] = np.uint8(0)
            raise

        records.append(
            {
                "timestamp_utc": datetime.utcnow().isoformat(),
                "product": product,
                "date": date,
                "date_index": date_index,
                "elapsed_seconds": time.perf_counter() - start_time,
                "status": "processed",
            }
        )

    return pd.DataFrame(records)


def append_processing_log(records):
    if records.empty:
        return

    if PROCESSING_LOG_FILE.exists():
        existing = pd.read_csv(
            PROCESSING_LOG_FILE,
            parse_dates=["date"],
        )

        records = pd.concat(
            [existing, records],
            ignore_index=True,
        )

        records = records.drop_duplicates(
            subset=["product", "date"],
            keep="last",
        )

    records.to_csv(
        PROCESSING_LOG_FILE,
        index=False,
    )

In [12]:
pilot_logs = []

pilot_dates_by_product = {}

for product in EXPECTED_PRODUCTS:
    product_dates = pd.DatetimeIndex(
        sorted(
            manifest.loc[
                manifest["product"].eq(product),
                "date",
            ].unique()
        )
    )

    pilot_dates = product_dates[:PILOT_DAYS]
    pilot_dates_by_product[product] = pilot_dates

    print(
        f"{product} pilot dates: "
        f"{pilot_dates.min().date()} to "
        f"{pilot_dates.max().date()}"
    )

    product_log = process_dates(
        product=product,
        dates_to_process=pilot_dates,
        skip_processed=True,
    )

    pilot_logs.append(product_log)

pilot_log = pd.concat(
    pilot_logs,
    ignore_index=True,
)

append_processing_log(pilot_log)

display(pilot_log)

print("FIVE-DAY PILOT PROCESSING: COMPLETE")

VNP46A1 pilot dates: 2013-05-12 to 2013-05-16


VNP46A1: 100%|██████████| 5/5 [00:00<00:00, 3456.08it/s]


VNP46A2 pilot dates: 2013-05-12 to 2013-05-16


VNP46A2: 100%|██████████| 5/5 [00:00<00:00, 2654.29it/s]


""


FIVE-DAY PILOT PROCESSING: COMPLETE


In [13]:
def source_sample(row, window):
    with rasterio.open(row["source_file"]) as src:
        sample = src.read(
            int(row["band_index"]),
            window=window,
            masked=True,
            out_dtype="float32",
        )

    sample = clean_source_block(sample[np.newaxis, ...])[0]

    return sample


qc_records = []

for product in EXPECTED_PRODUCTS:
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    date_lookup = {
        pd.Timestamp(date): index
        for index, date in enumerate(dates)
    }

    root = zarr.open_group(
        str(STORE_PATHS[product]),
        mode="r",
    )

    height = int(root.attrs["height"])
    width = int(root.attrs["width"])

    sample_height = min(QC_WINDOW_SIZE, height)
    sample_width = min(QC_WINDOW_SIZE, width)

    row_off = max((height - sample_height) // 2, 0)
    col_off = max((width - sample_width) // 2, 0)

    audit_window = Window(
        col_off=col_off,
        row_off=row_off,
        width=sample_width,
        height=sample_height,
    )

    for date in pilot_dates_by_product[product]:
        date = pd.Timestamp(date).normalize()
        date_index = date_lookup[date]

        if root["processed"][date_index] != 1:
            raise RuntimeError(
                f"{product} {date.date()} was not marked processed."
            )

        date_rows = product_manifest[
            product_manifest["date"].eq(date)
        ].sort_values("variable")

        for _, row in date_rows.iterrows():
            variable = row["variable"]

            source_values = source_sample(
                row,
                audit_window,
            )

            stored_values = root[variable][
                date_index,
                row_off:row_off + sample_height,
                col_off:col_off + sample_width,
            ]

            np.testing.assert_allclose(
                stored_values,
                source_values,
                rtol=0,
                atol=0,
                equal_nan=True,
                err_msg=(
                    f"Source-to-store mismatch: "
                    f"{product}, {date.date()}, {variable}"
                ),
            )

            sampled = root[variable][
                date_index,
                ::QC_SAMPLE_STEP,
                ::QC_SAMPLE_STEP,
            ]

            finite = sampled[np.isfinite(sampled)]

            is_flag_band = any(
                token in variable.lower()
                for token in (
                    "flag",
                    "quality",
                    "qf_",
                    "cloud_mask",
                )
            )

            integer_like = True

            if is_flag_band and finite.size:
                integer_like = np.allclose(
                    finite,
                    np.round(finite),
                    rtol=0,
                    atol=1e-6,
                )

            qc_records.append(
                {
                    "product": product,
                    "date": date,
                    "variable": variable,
                    "sample_pixels": sampled.size,
                    "finite_pixels": finite.size,
                    "finite_fraction": (
                        finite.size / sampled.size
                        if sampled.size
                        else np.nan
                    ),
                    "minimum": (
                        float(np.min(finite))
                        if finite.size
                        else np.nan
                    ),
                    "median": (
                        float(np.median(finite))
                        if finite.size
                        else np.nan
                    ),
                    "maximum": (
                        float(np.max(finite))
                        if finite.size
                        else np.nan
                    ),
                    "flag_band": is_flag_band,
                    "integer_like": integer_like,
                    "source_match": True,
                }
            )

pilot_qc = pd.DataFrame(qc_records)

all_missing_variables = (
    pilot_qc.groupby(["product", "variable"])["finite_pixels"]
    .sum()
    .loc[lambda values: values == 0]
)

noninteger_flags = pilot_qc[
    pilot_qc["flag_band"]
    & ~pilot_qc["integer_like"]
]

if not all_missing_variables.empty:
    display(all_missing_variables)

    raise ValueError(
        "One or more variables contain no finite pixels "
        "across the five-day pilot."
    )

if not noninteger_flags.empty:
    display(noninteger_flags)

    raise ValueError(
        "A flag or quality band contains non-integer values."
    )

pilot_qc.to_csv(
    PILOT_QC_FILE,
    index=False,
)

pilot_pass = {
    "status": "passed",
    "timestamp_utc": datetime.utcnow().isoformat(),
    "pilot_days": PILOT_DAYS,
    "products": list(EXPECTED_PRODUCTS),
    "source_manifest": str(MANIFEST_FILE),
    "pilot_qc": str(PILOT_QC_FILE),
    "manifest_signatures": {
        product: stores[product].attrs["manifest_signature"]
        for product in EXPECTED_PRODUCTS
    },
}

PILOT_PASS_FILE.write_text(
    json.dumps(
        pilot_pass,
        indent=2,
    )
)

display(
    pilot_qc.groupby(["product", "variable"])
    .agg(
        pilot_finite_fraction=("finite_fraction", "mean"),
        pilot_minimum=("minimum", "min"),
        pilot_median=("median", "median"),
        pilot_maximum=("maximum", "max"),
        source_matches=("source_match", "all"),
    )
)

print("PILOT SANITY CHECKS: PASSED")
print(f"QC report: {PILOT_QC_FILE}")
print(f"Batch authorisation: {PILOT_PASS_FILE}")

pilot_finite_fraction  \
product variable                                                   
VNP46A1 DNB_At_Sensor_Radiance_500m                     0.319608   
        Granule                                         0.319608   
        Moon_Illumination_Fraction                      0.319608   
        QF_Cloud_Mask                                   0.319608   
        QF_DNB                                          0.319608   
        Sensor_Azimuth                                  0.319608   
        Sensor_Zenith                                   0.319608   
        UTC_Time                                        0.319608   
VNP46A2 DNB_BRDF_Corrected_NTL                          0.128235   
        DNB_Lunar_Irradiance                            0.319529   
        Gap_Filled_DNB_BRDF_Corrected_NTL               0.319608   
        Latest_High_Quality_Retrieval                   0.319608   
        Mandatory_Quality_Flag                          0.128235   
        QF_Cloud_Mask                                   0.319529   
        Snow_Flag                                       0.319608   

                                           pilot_minimum  pilot_median  \
product variable                                                         
VNP46A1 DNB_At_Sensor_Radiance_500m             0.000000      0.400000   
        Granule                                 0.000000      1.000000   
        Moon_Illumination_Fraction              6.590000     19.030001   
        QF_Cloud_Mask                          34.000000    738.000000   
        QF_DNB                                  0.000000      0.000000   
        Sensor_Azimuth                       -153.729996    -78.440002   
        Sensor_Zenith                           0.020000     47.860001   
        UTC_Time                               16.310972     17.251501   
VNP46A2 DNB_BRDF_Corrected_NTL                  0.006799      0.335258   
        DNB_Lunar_Irradiance                    0.500000      0.500000   
        Gap_Filled_DNB_BRDF_Corrected_NTL       0.006799      0.208544   
        Latest_High_Quality_Retrieval           0.000000      5.000000   
        Mandatory_Quality_Flag                  0.000000      0.000000   
        QF_Cloud_Mask                          34.000000    738.000000   
        Snow_Flag                               0.000000      0.000000   

                                           pilot_maximum  source_matches  
product variable                                                          
VNP46A1 DNB_At_Sensor_Radiance_500m            75.199997            True  
        Granule                                 2.000000            True  
        Moon_Illumination_Fraction             35.490002            True  
        QF_Cloud_Mask                         762.000000            True  
        QF_DNB                                  0.000000            True  
        Sensor_Azimuth                        176.419998            True  
        Sensor_Zenith                          66.309998            True  
        UTC_Time                               17.989864            True  
VNP46A2 DNB_BRDF_Corrected_NTL                 82.273148            True  
        DNB_Lunar_Irradiance                    0.500000            True  
        Gap_Filled_DNB_BRDF_Corrected_NTL      82.273148            True  
        Latest_High_Quality_Retrieval          12.000000            True  
        Mandatory_Quality_Flag                  1.000000            True  
        QF_Cloud_Mask                        2146.000000            True  
        Snow_Flag                               1.000000            True

PILOT SANITY CHECKS: PASSED
QC report: ../datasets/VNP46/processed/pilot_qc.csv
Batch authorisation: ../datasets/VNP46/processed/pilot_qc_passed.json


In [14]:
status_records = []

for product in EXPECTED_PRODUCTS:
    root = zarr.open_group(
        str(STORE_PATHS[product]),
        mode="r",
    )

    processed = root["processed"][:] == 1

    status_records.append(
        {
            "product": product,
            "processed_dates": int(processed.sum()),
            "total_dates": int(processed.size),
            "remaining_dates": int((~processed).sum()),
            "completion_pct": 100 * processed.mean(),
            "store": str(STORE_PATHS[product]),
        }
    )

processing_status = pd.DataFrame(status_records)

display(processing_status)

,product,processed_dates,total_dates,remaining_dates,completion_pct,store
0,VNP46A1,5,546,541,0.915751,../datasets/VNP46/processed/Haiyan_VNP46A1.zarr
1,VNP46A2,5,546,541,0.915751,../datasets/VNP46/processed/Haiyan_VNP46A2.zarr


In [15]:
RUN_FULL_BATCH = True


if RUN_FULL_BATCH:
    if not PILOT_PASS_FILE.exists():
        raise RuntimeError(
            "Full processing is blocked because the five-day "
            "pilot has not passed QC."
        )

    with PILOT_PASS_FILE.open() as file:
        pilot_authorisation = json.load(file)

    if pilot_authorisation.get("status") != "passed":
        raise RuntimeError(
            "The pilot QC authorisation is invalid."
        )

    batch_logs = []

    for product in EXPECTED_PRODUCTS:
        product_dates = pd.DatetimeIndex(
            sorted(
                manifest.loc[
                    manifest["product"].eq(product),
                    "date",
                ].unique()
            )
        )

        product_log = process_dates(
            product=product,
            dates_to_process=product_dates,
            skip_processed=True,
        )

        batch_logs.append(product_log)

    batch_log = pd.concat(
        batch_logs,
        ignore_index=True,
    )

    append_processing_log(batch_log)

    final_status = []

    for product in EXPECTED_PRODUCTS:
        root = zarr.open_group(
            str(STORE_PATHS[product]),
            mode="r",
        )

        processed = root["processed"][:] == 1

        final_status.append(
            {
                "product": product,
                "processed_dates": int(processed.sum()),
                "total_dates": int(processed.size),
                "remaining_dates": int((~processed).sum()),
                "complete": bool(processed.all()),
            }
        )

    final_status = pd.DataFrame(final_status)
    display(final_status)

    if not final_status["complete"].all():
        raise RuntimeError(
            "The batch ended with unprocessed dates. "
            "Rerun this cell to resume."
        )

    for store_path in STORE_PATHS.values():
        zarr.consolidate_metadata(str(store_path))

    print("FULL BATCH PROCESSING: COMPLETE")

else:
    print(
        "Full batch not started. "
        "Set RUN_FULL_BATCH = True after reviewing the pilot QC."
    )

VNP46A2: 100%|██████████| 546/546 [24:21<00:00,  2.68s/it]


,product,processed_dates,total_dates,remaining_dates,complete
0,VNP46A1,546,546,0,True
1,VNP46A2,546,546,0,True


FULL BATCH PROCESSING: COMPLETE
